# REPSOL Pretrained Model Evaluation (70/15/15)

**Pretrained Model**

This notebook runs the evaluation workflow for the pretrained checkpoint:
1. Configure evaluation hyperparameters.
2. Verify spectrogram tensor files in train/val/test.
3. Load the pretrained checkpoint.
4. Evaluate on validation and test sets.
5. Print detailed classification report and confusion matrix.

In [1]:
from pathlib import Path
import torch

# ===== Hyperparameters (edit these) =====
BATCH_SIZE = 8
MODEL_NAME = "efficientnet"

# ===== Paths =====
PROJECT_ROOT = Path(r"D:\Work\Internships\INMAR\REPSOL")
SPECTROGRAM_DIR = PROJECT_ROOT / "Data" / "Spectrograms"
OUTPUT_DIR = PROJECT_ROOT / "Models_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = OUTPUT_DIR / "PreTrained_model_best_01.pth"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SPECTROGRAM_DIR:", SPECTROGRAM_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("DEVICE:", DEVICE)
print("BATCH_SIZE:", BATCH_SIZE)

PROJECT_ROOT: D:\Work\Internships\INMAR\REPSOL
SPECTROGRAM_DIR: D:\Work\Internships\INMAR\REPSOL\Data\Spectrograms
OUTPUT_DIR: D:\Work\Internships\INMAR\REPSOL\Models_output
CHECKPOINT_PATH: D:\Work\Internships\INMAR\REPSOL\Models_output\PreTrained_model_best_01.pth
DEVICE: cpu
BATCH_SIZE: 8


In [2]:
def count_pt_files(root):
    counts = {}
    for split in ["train", "val", "test"]:
        split_dir = root / split
        counts[split] = sum(1 for _ in split_dir.rglob("*.pt")) if split_dir.exists() else 0
    return counts

counts = count_pt_files(SPECTROGRAM_DIR)
print("PT files by split:", counts)
print("Total:", sum(counts.values()))

assert counts["train"] > 0, "No train .pt files found."
assert counts["val"] > 0, "No val .pt files found."
assert counts["test"] > 0, "No test .pt files found."

PT files by split: {'train': 1382, 'val': 296, 'test': 297}
Total: 1975


In [3]:
# Install/verify evaluation dependencies in the active notebook kernel
import importlib
import subprocess
import sys

required = ["torch", "torchvision", "torchaudio", "scikit-learn", "pandas", "tqdm", "numpy"]
name_map = {
    "scikit-learn": "sklearn",
}

for pkg in required:
    import_name = name_map.get(pkg, pkg.replace("-", "_"))
    try:
        importlib.import_module(import_name)
        print(f"OK: {pkg}")
    except Exception:
        print(f"Installing: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        print(f"Installed: {pkg}")

OK: torch
OK: torchvision
OK: torchaudio
OK: scikit-learn
OK: pandas
OK: tqdm
OK: numpy


In [8]:
import importlib
import sys
import torch

try:
    import torchvision  # noqa: F401
except Exception as e:
    raise RuntimeError(
        "torchvision is not available in this notebook kernel. "
        "Run the dependency cell right above this one, then retry."
    ) from e

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.PreTrained_model.load_pretrained as load_module
import src.dataloaders as dataloaders_module
import src.evaluate as eval_module

load_module = importlib.reload(load_module)
dataloaders_module = importlib.reload(dataloaders_module)
eval_module = importlib.reload(eval_module)

load_model = load_module.load_model
get_dataloaders = dataloaders_module.get_dataloaders
evaluate_model = eval_module.evaluate_model

assert CHECKPOINT_PATH.exists(), f"Checkpoint not found: {CHECKPOINT_PATH}"

model, _ = load_model(
    str(CHECKPOINT_PATH),
    model_name=MODEL_NAME,
    num_classes=None,
    device=DEVICE,
    freeze_backbone=False,
)

print("Model loaded from:", CHECKPOINT_PATH)
print("Model device:", DEVICE) 

_, val_loader, test_loader = get_dataloaders(
    SPECTROGRAM_DIR,
    batch_size=BATCH_SIZE,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=False,
    prefetch_factor=2,
    cache_in_memory=False,
)

print("Validation loader length:", len(val_loader))
print("Test loader length:", len(test_loader))

Model loaded from: D:\Work\Internships\INMAR\REPSOL\Models_output\PreTrained_model_best_01.pth
Model device: cpu
Validation loader length: 37
Test loader length: 38


In [9]:
print("Running validation evaluation...")
val_metrics = evaluate_model(model, val_loader, DEVICE)
print("Validation evaluation complete.")

print("Running test evaluation...")
test_metrics = evaluate_model(model, test_loader, DEVICE)
print("Test evaluation complete.")

print("\nValidation Metrics")
print({
    "accuracy": round(val_metrics["accuracy"], 4),
    "precision": round(val_metrics["precision"], 4),
    "recall": round(val_metrics["recall"], 4),
    "f1": round(val_metrics["f1"], 4),
})

print("\nTest Metrics")
print({
    "accuracy": round(test_metrics["accuracy"], 4),
    "precision": round(test_metrics["precision"], 4),
    "recall": round(test_metrics["recall"], 4),
    "f1": round(test_metrics["f1"], 4),
})

Running validation evaluation...
Validation evaluation complete.
Running test evaluation...
Test evaluation complete.

Validation Metrics
{'accuracy': 0.0034, 'precision': 0.2534, 'recall': 0.0034, 'f1': 0.0067}

Test Metrics
{'accuracy': 0.0034, 'precision': 0.037, 'recall': 0.0034, 'f1': 0.0062}


In [10]:
print("Test Classification Report:\n")
print(test_metrics["report"])

print("\nTest Confusion Matrix:")
print(test_metrics["confusion_matrix"])

Test Classification Report:

              precision    recall  f1-score   support

           0       0.33      0.03      0.06        33
           1       0.00      0.00      0.00         6
           2       0.00      0.00      0.00        20
           3       0.00      0.00      0.00       106
           4       0.00      0.00      0.00        39
           5       0.00      0.00      0.00        11
           6       0.00      0.00      0.00        76
           7       0.00      0.00      0.00         6
          10       0.00      0.00      0.00         0
          12       0.00      0.00      0.00         0
          13       0.00      0.00      0.00         0
          18       0.00      0.00      0.00         0
          22       0.00      0.00      0.00         0
          25       0.00      0.00      0.00         0
          27       0.00      0.00      0.00         0
          29       0.00      0.00      0.00         0
          30       0.00      0.00      0.00         